In [1]:
import pickle

# from tuning_utils import tune_over_years
from training_utils import train_gplm, train_gpb, train_ilm, train_itb

import sys
# sys.path.append("PATH TO loglikelihood_utils")

from QB_utils import QB_nll
from ZOCN_utils import ZOCN_nll
from BEINF_utils import BEINF_nll
from ZOCTN_utils import ZOCTN_nll
from ZOCSG_utils import ZOCSG_nll
from ZOCTB_utils import ZOCTB_nll

/Users/christopher/Documents/University/ETH MScQF/Term 3/Thesis/working_project/mscqf_thesis/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/Users/christopher/Documents/University/ETH MScQF/Term 3/Thesis/working_project/mscqf_thesis/lib/python3.12/site-packages/keras/src/export/tf2onnx_lib.py:8: FutureWarning: In the future `np.object` will be defined as the corresponding NumPy scalar.
  if not hasattr(np, "object"):


In [2]:
sfeature_cols = ["X_centroid", "Y_centroid", "credit_score", "occupancy", "nr_units", "loan_purpose", "first_time_homebuyer", "MSA", 
                "insurance_percent", "original_debt_to_income", "original_loan_to_value", "original_upb", "number_of_borrowers", "ir_spread", 
                "n_months", "ltv_at_default", "gdp_growth", "ln_income_per_capita", "ln_expenditures_per_capita", "unemployment_rate", 
                "hpi_growth", "gdp_construction_growth", "gross_operating_surplus_growth", "inflation_rate", "DJIA_growth"]
stfeature_cols = ["year_default", "X_centroid", "Y_centroid", "credit_score", "occupancy", "nr_units", "loan_purpose", "first_time_homebuyer", "MSA", 
                "insurance_percent", "original_debt_to_income", "original_loan_to_value", "original_upb", "number_of_borrowers", "ir_spread", 
                "n_months", "ltv_at_default", "gdp_growth", "ln_income_per_capita", "ln_expenditures_per_capita", "unemployment_rate", 
                "hpi_growth", "gdp_construction_growth", "gross_operating_surplus_growth", "inflation_rate", "DJIA_growth"]
target_col = ["LGD"]
scoords_cols = ["X_centroid", "Y_centroid"]
stcoords_cols = ["year_default", "X_centroid", "Y_centroid"]

categorical_cols = ["occupancy", "loan_purpose", "first_time_homebuyer", "MSA", "number_of_borrowers"]

year_range = [i for i in range(2008, 2023)]

# Tuning

In [ ]:
# Tune ITB Models
likelihood = 'zoctn' # OR 'zero_one_censored_shifted_gamma' OR 'binomial_logit'
tune_over_years('ind', likelihood, year_range, sfeature_cols, categorical_cols, target_col)

# Tune SGPB models
tune_over_years('gp', 'zoctn', year_range, sfeature_cols, categorical_cols, target_col, location_type="spatial", coord_cols=scoords_cols)

# Tune STGPB models
tune_over_years('gp', 'zoctn', year_range, stfeature_cols, categorical_cols, target_col, location_type="spatiotemporal", coord_cols=stcoords_cols)

# Training

In [3]:
# Training ILMs
ilm_nlls = {
    "ZOCTN": {"obj": ZOCTN_nll, 'N_aux_params': 3},
    "ZOCSG": {"obj": ZOCSG_nll, 'N_aux_params': 2},
    "ZOCTB": {"obj": ZOCTB_nll, 'N_aux_params': 2},
    "QB": {"obj": QB_nll, 'N_aux_params': 0},
    "ZOCN": {"obj": ZOCN_nll, 'N_aux_params': 1},
    "BEINF": {"obj": BEINF_nll, 'N_aux_params': 3}
}
for y in year_range:
    train_ilm(ilm_nlls, y, sfeature_cols, target_col, categorical_cols)


# Training ITBs
itb_nlls = {
    "zoctn": {
        "N_aux_params": 3,
    },
    "zero_one_censored_shifted_gamma": {
        "N_aux_params": 2,
    },
    "binomial_logit": {
        "N_aux_params": 0,
    },
}
for y in year_range:
    for model in itb_nlls:
        tuning_dict = {}
        with open(f'tuned_parameters/ind_{model}_hyperparams_{y}.pickle', 'rb') as handle:
            tuning_dict = pickle.load(handle)

        itb_nlls[model]['hyperparams'] = tuning_dict['best_params']
        itb_nlls[model]['N_rounds'] = tuning_dict['best_iter']
    
    train_itb(itb_nlls, y, sfeature_cols, target_col, categorical_cols)


# Training GPLMs
for y in year_range:
    # SGPLM
    train_gplm('spatial', 'zoctn', y, sfeature_cols, target_col, scoords_cols, categorical_cols)

    # STGPLM
    train_gplm('spatiotemporal', 'zoctn', y, stfeature_cols, target_col, stcoords_cols, categorical_cols)    


# Training GPBs
for y in year_range:
    # SGPB
    tuning_dict = {}
    with open(f'tuned_parameters/spatialgp_zoctn_hyperparams_{y}.pickle', 'rb') as handle:
        tuning_dict = pickle.load(handle)

    hps, N_rounds = tuning_dict['best_params'], tuning_dict['best_iter']
    train_gpb('spatial', 'zoctn', y, sfeature_cols, target_col, scoords_cols, categorical_cols, hps, N_rounds)

    # STGPB
    tuning_dict = {}
    with open(f'tuned_parameters/spatiotemporalgp_zoctn_hyperparams_{y}.pickle', 'rb') as handle:
        tuning_dict = pickle.load(handle)

    hps, N_rounds = tuning_dict['best_params'], tuning_dict['best_iter']
    train_gpb('spatiotemporal', 'zoctn', y, stfeature_cols, target_col, stcoords_cols, categorical_cols)
